# GoNative Suitability Scorer
## Gary Ding &nbsp;&nbsp;&nbsp; March 3rd, 2026

Queries the Go Native Plants database (gonativeplants.org) to retrieve all species
matching required site constraints, then computes an equal-weight multi-criteria
suitability score based on preferred landscape and horticultural attributes.

---

## Dependencies

In [ ]:
!pip install requests beautifulsoup4 openpyxl -q
print("Ready.")

Ready.


## Configuration

Required filters define the candidate pool. Preferred filters define the scoring criteria.
Modify filter codes as needed using the Developer Tools method (F12 → Network → Payload).

In [12]:
# ════════════════════════════════════════════════════════════
# REQUIRED FILTERS — define the candidate plant pool
# ════════════════════════════════════════════════════════════
REQUIRED_FILTERS = {
    "plant_form":         ["tree", "shrub"],
    "islands":            ["hawaii"],
    "water_requirements": ["1.00", "3.00"],   # range: [min, max]
}

# ════════════════════════════════════════════════════════════
# PREFERRED FILTERS — scored attributes (equal weight)
# ════════════════════════════════════════════════════════════
# Each entry: (facet_name, {facet_value: points, ...})
# Points encode ordinal preference within each criterion.
PREFERRED_FILTERS = [
    ("short_lived",      {"0": 1}),                             # binary
    ("good_for_novices", {"easy": 2, "average": 1}),            # ordinal
    ("availability",     {"common": 2, "variable": 1}),         # ordinal
    # ("climate_zone",     {"ca": 1, "ia": 1, "id": 1, "im": 1}), # cumulative
    ("water_requirements", {("1.00", "2.00"): 1}),              # range: drought tolerant bonus
    ("spreader",         {"spreader": 2, "variable": 1}),       # ordinal
    ("pruning_needs", {"regular": 2, "minimal": 1}),            # ordinal
]

OUTPUT_FILE = "gonative_suitability_scored.xlsx"

print("Configuration set.")

Configuration set.


## Execute process

In [13]:
import requests
import json
import re
import time
import html as html_lib
from datetime import datetime
from bs4 import BeautifulSoup
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side, numbers
from google.colab import files

API_URL = "https://www.gonativeplants.org/wp-json/facetwp/v1/refresh"
PAGE_URL = "https://www.gonativeplants.org/gonativeplants/"
REQUEST_DELAY = 1.0  # seconds between API calls


# ── Session / nonce ──

def init_session():
    """Establish session and extract FacetWP nonce from page source."""
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/145.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
    })
    resp = session.get(PAGE_URL, timeout=30)
    resp.raise_for_status()

    nonce_match = re.search(
        r'FWP_JSON\s*=\s*\{[^}]*?"nonce"\s*:\s*"([a-f0-9]+)"', resp.text
    )
    if not nonce_match:
        nonce_match = re.search(r'"nonce"\s*:\s*"([a-f0-9]{8,})"', resp.text)
    if not nonce_match:
        raise RuntimeError("FacetWP nonce not found in page source.")

    nonce = nonce_match.group(1)
    session.headers.update({
        "Content-Type": "application/json",
        "Accept": "*/*",
        "Origin": "https://www.gonativeplants.org",
        "Referer": PAGE_URL,
        "X-WP-Nonce": nonce,
    })
    print(f"Session initialized. Nonce: {nonce}")
    return session


# ── API interface ──

ALL_FACET_KEYS = [
    "keyword_search", "climate_zone", "exposure", "water_requirements",
    "well_drained_soil", "salt_tolerant", "elevation", "plant_form",
    "landscape_uses", "plant_height", "plant_width", "flower_color",
    "good_for_novices", "spreader", "degree_of_spread", "short_lived",
    "pruning_needs", "availability", "ethnobotany_uses", "islands",
    "attracted_fauna", "valued_by_bees", "federal_status",
    "population_status", "native_status",
    "go_native_results_count", "go_native_pagination",
    "go_native_posts_per_page", "sort_by_name",
]

def build_payload(filters, page=1):
    facets = {k: filters.get(k, [] if k != "keyword_search" else "") for k in ALL_FACET_KEYS}
    facets["keyword_search"] = ""
    return {
        "action": "facetwp_refresh",
        "data": {
            "facets": facets,
            "frozen_facets": {},
            "http_params": {"get": {}, "uri": "gonativeplants", "url_vars": {}},
            "template": "go_native_plants_results",
            "extras": {"selections": True, "sort": "default"},
            "soft_refresh": 0, "is_bfcache": 0, "first_load": 0,
            "paged": page,
        },
    }


def parse_cards(html):
    """Extract plant records from FacetWP template HTML.
    Returns list of dicts keyed by post_id."""
    soup = BeautifulSoup(html, "html.parser")
    plants = []
    for card in soup.select("a.aka_type_query_item"):
        post_id = card.get("data-post___id", "")
        title_raw = html_lib.unescape(card.get("data-post___title", ""))
        link = card.get("data-post___link", "")

        common_el = card.select_one("span.gn_common_name")
        sci_el = card.select_one("span.gn_scientific_name")
        hawaiian = common_el.get_text(strip=True) if common_el else ""
        scientific = sci_el.get_text(strip=True) if sci_el else ""

        if not hawaiian and title_raw:
            m = re.match(r'^(.+?)\s*\((.+)\)\s*$', title_raw)
            if m:
                hawaiian, scientific = m.group(1).strip(), m.group(2).strip()
            else:
                hawaiian = title_raw

        img = card.select_one("figure img, img.wp-post-image")
        img_url = (img.get("src", "") if img else "")

        plants.append({
            "post_id": post_id,
            "hawaiian_name": hawaiian,
            "scientific_name": scientific,
            "detail_url": link,
            "image_url": img_url,
        })
    return plants


def get_total_pages(data):
    tp = 1
    for path in [
        lambda d: d.get("settings", {}).get("pager", {}).get("total_pages", 1),
        lambda d: d.get("pager", {}).get("total_pages", 1),
    ]:
        val = path(data)
        if isinstance(val, (int, float)) and val > tp:
            tp = int(val)
    pg_html = data.get("facets", {}).get("go_native_pagination", "")
    if isinstance(pg_html, str):
        nums = re.findall(r'data-page="(\d+)"', pg_html)
        if nums:
            tp = max(tp, max(int(n) for n in nums))
    return tp


def fetch_plant_ids(session, filters, label=""):
    """Fetch all post_ids matching the given filter set.
    Returns (set of post_ids, list of plant dicts on first call)."""
    all_ids = set()
    all_plants = []
    page = 1
    total_pages = None

    while True:
        payload = build_payload(filters, page)
        resp = session.post(API_URL, json=payload, timeout=30)
        resp.raise_for_status()
        data = resp.json()

        if page == 1:
            total_pages = get_total_pages(data)

        html = data.get("template", "")
        if not html:
            break

        plants = parse_cards(html)
        if not plants:
            break

        for p in plants:
            all_ids.add(p["post_id"])
        all_plants.extend(plants)

        if total_pages is None or page >= total_pages:
            break
        page += 1
        time.sleep(REQUEST_DELAY)

    tag = f" [{label}]" if label else ""
    print(f"  {len(all_ids)} plants across {page} page(s){tag}")
    return all_ids, all_plants


# ── Scoring engine ──

def compute_scores(session, base_filters, base_ids, preferred_filters):
    """Equal-weight additive multi-criteria scoring.

    Model: S = (1/K) * sum_k [ s_k ] * 100  where
        k   = preferred criterion index (1..K)
        s_k = normalized score for criterion k in [0, 1]

    All criteria receive equal weight (1/K). Within each criterion,
    raw points are normalized to [0, 1] by dividing by the maximum
    attainable points. For cumulative criteria (e.g., climate zones),
    the score equals the fraction of target values matched.

    The composite score is rescaled to [0, 100] for readability.
    """
    K = len(preferred_filters)

    # Equal weights
    roc_weights = [1.0 / K] * K

    print(f"\nScoring model: {K} criteria, equal weights (1/{K} = {1.0/K:.4f} each):")
    for i, (facet_name, value_map) in enumerate(preferred_filters):
        print(f"  {facet_name}: values={value_map}")

    # Initialize scores
    scores = {pid: [0.0] * K for pid in base_ids}

    # Query each preferred criterion
    print(f"\nQuerying preferred criteria...")
    for k, (facet_name, value_map) in enumerate(preferred_filters):
        max_points = max(value_map.values())

        for facet_val, points in value_map.items():
            # Build filter set: required + this preferred value
            probe_filters = dict(base_filters)
            if isinstance(facet_val, tuple):
              probe_filters[facet_name] = list(facet_val)  # sends ["1.00", "2.00"] as range
            else:
              probe_filters[facet_name] = [facet_val]       # existing single-value logic

            time.sleep(REQUEST_DELAY)
            matched_ids, _ = fetch_plant_ids(
                session, probe_filters,
                label=f"{facet_name}={facet_val}"
            )

            # Assign points to plants in the base set that matched
            for pid in matched_ids:
                if pid in scores:
                    if facet_name == "climate_zone":
                        # Cumulative: add points for each zone matched
                        scores[pid][k] += points
                    else:
                        # Ordinal/binary: take maximum points
                        scores[pid][k] = max(scores[pid][k], points)

        # Determine max_points for normalization
        if facet_name == "climate_zone":
            # Max = sum of all zone points
            max_points = sum(value_map.values())

        # Normalize criterion scores to [0, 1]
        for pid in scores:
            scores[pid][k] = scores[pid][k] / max_points if max_points > 0 else 0.0

    # Compute composite scores
    composite = {}
    for pid in scores:
        raw = sum(roc_weights[k] * scores[pid][k] for k in range(K))
        composite[pid] = round(raw * 100, 2)

    return composite, scores, roc_weights


# ── XLSX output ──

def build_xlsx(plants, composite, criterion_scores, roc_weights,
               preferred_filters, required_filters, filename):
    wb = Workbook()

    # --- Sheet 1: Scored Plant List ---
    ws = wb.active
    ws.title = "Scored Plant List"

    hdr_font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
    hdr_fill = PatternFill("solid", fgColor="2E7D32")
    hdr_align = Alignment(horizontal="center", wrap_text=True)
    data_font = Font(name="Arial", size=10)
    score_font = Font(name="Arial", size=10, bold=True)
    thin_border = Border(
        bottom=Side(style="thin", color="CCCCCC")
    )

    # Conditional fill for score tiers
    fill_high = PatternFill("solid", fgColor="C8E6C9")     # >=70
    fill_med  = PatternFill("solid", fgColor="FFF9C4")     # >=40
    fill_low  = PatternFill("solid", fgColor="FFCDD2")     # <40

    # Column definitions
    K = len(preferred_filters)
    base_cols = [
        ("Rank", 6),
        ("Suitability\nScore", 12),
        ("Hawaiian Name", 22),
        ("Scientific Name", 35),
    ]
    pref_cols = [(f"{name.replace('_',' ').title()}", 14) for i, (name, _) in enumerate(preferred_filters)]
    extra_cols = [("Detail URL", 50)]
    all_cols = base_cols + pref_cols + extra_cols

    for col, (label, width) in enumerate(all_cols, 1):
        cell = ws.cell(row=1, column=col, value=label)
        cell.font = hdr_font
        cell.fill = hdr_fill
        cell.alignment = hdr_align
        ws.column_dimensions[cell.column_letter].width = width

    # Sort plants by composite score descending
    sorted_plants = sorted(plants, key=lambda p: composite.get(p["post_id"], 0), reverse=True)

    for row_idx, plant in enumerate(sorted_plants, 2):
        pid = plant["post_id"]
        score = composite.get(pid, 0)
        crit = criterion_scores.get(pid, [0.0] * K)

        col = 1
        # Rank
        ws.cell(row=row_idx, column=col, value=row_idx - 1).font = data_font
        col += 1
        # Composite score
        score_cell = ws.cell(row=row_idx, column=col, value=score)
        score_cell.font = score_font
        score_cell.number_format = '0.00'
        if score >= 70:
            score_cell.fill = fill_high
        elif score >= 40:
            score_cell.fill = fill_med
        else:
            score_cell.fill = fill_low
        col += 1
        # Names
        ws.cell(row=row_idx, column=col, value=plant["hawaiian_name"]).font = data_font
        col += 1
        sci_cell = ws.cell(row=row_idx, column=col, value=plant["scientific_name"])
        sci_cell.font = Font(name="Arial", size=10, italic=True)
        col += 1
        # Criterion sub-scores (normalized 0-1, displayed as %)
        for k in range(K):
            c = ws.cell(row=row_idx, column=col, value=crit[k])
            c.font = data_font
            c.number_format = '0%'
            c.alignment = Alignment(horizontal="center")
            col += 1
        # URL
        ws.cell(row=row_idx, column=col, value=plant["detail_url"]).font = data_font

        # Row border
        for c in range(1, len(all_cols) + 1):
            ws.cell(row=row_idx, column=c).border = thin_border

    # Freeze header row
    ws.freeze_panes = "A2"
    # Auto-filter
    ws.auto_filter.ref = f"A1:{chr(64 + len(all_cols))}{len(sorted_plants) + 1}"

    # --- Sheet 2: Methodology ---
    ws2 = wb.create_sheet("Methodology")
    ws2.column_dimensions["A"].width = 25
    ws2.column_dimensions["B"].width = 60
    ws2.column_dimensions["C"].width = 15

    title_font = Font(bold=True, name="Arial", size=12)
    section_font = Font(bold=True, name="Arial", size=10)
    body_font = Font(name="Arial", size=10)

    r = 1
    ws2.cell(row=r, column=1, value="GoNative Suitability Scoring Methodology").font = title_font
    r += 2

    ws2.cell(row=r, column=1, value="Scoring Model").font = section_font
    r += 1
    ws2.cell(row=r, column=1, value="S = (1/K) · Σ sk × 100").font = body_font
    r += 1
    ws2.cell(row=r, column=1, value="where K = number of criteria, sk = normalized criterion score [0,1]").font = body_font
    r += 2

    ws2.cell(row=r, column=1, value="Weight Derivation").font = section_font
    r += 1
    ws2.cell(row=r, column=1, value="Equal weighting:").font = body_font
    ws2.cell(row=r, column=2, value="All criteria receive weight 1/K. Score reflects the fraction of preferred criteria satisfied.").font = body_font
    r += 2

    ws2.cell(row=r, column=1, value="Criterion Weights").font = section_font
    r += 1
    ws2.cell(row=r, column=1, value="Criterion").font = hdr_font
    ws2.cell(row=r, column=1).fill = hdr_fill
    ws2.cell(row=r, column=2, value="Values").font = hdr_font
    ws2.cell(row=r, column=2).fill = hdr_fill
    ws2.cell(row=r, column=3, value="Weight").font = hdr_font
    ws2.cell(row=r, column=3).fill = hdr_fill
    r += 1
    for i, (name, vmap) in enumerate(preferred_filters):
        vals_str = ", ".join(f"{v}={p}pt" for v, p in vmap.items())
        ws2.cell(row=r, column=1, value=name.replace('_',' ').title()).font = body_font
        ws2.cell(row=r, column=2, value=vals_str).font = body_font
        ws2.cell(row=r, column=3, value=roc_weights[i]).font = body_font
        ws2.cell(row=r, column=3).number_format = '0.0000'
        r += 1
    r += 1

    ws2.cell(row=r, column=1, value="Score Interpretation").font = section_font
    r += 1
    for label, desc in [
        ("≥ 70 (green)", "High suitability: meets most preferred criteria"),
        ("40–69 (yellow)", "Moderate suitability: meets some preferred criteria"),
        ("< 40 (red)", "Low suitability: meets few preferred criteria"),
    ]:
        ws2.cell(row=r, column=1, value=label).font = body_font
        ws2.cell(row=r, column=2, value=desc).font = body_font
        r += 1
    r += 1

    ws2.cell(row=r, column=1, value="Required Filters").font = section_font
    r += 1
    for k, v in required_filters.items():
        ws2.cell(row=r, column=1, value=k.replace('_', ' ').title()).font = body_font
        ws2.cell(row=r, column=2, value=", ".join(v)).font = body_font
        r += 1
    r += 1

    ws2.cell(row=r, column=1, value="Data Source").font = section_font
    r += 1
    ws2.cell(row=r, column=1, value="Go Native Plants").font = body_font
    ws2.cell(row=r, column=2, value="https://www.gonativeplants.org/gonativeplants/").font = body_font
    r += 1
    ws2.cell(row=r, column=1, value="Generated").font = body_font
    ws2.cell(row=r, column=2, value=datetime.now().strftime('%Y-%m-%d %H:%M')).font = body_font

    wb.save(filename)
    return True


# ═══════════════════════════════════
# Main execution
# ═══════════════════════════════════

session = init_session()

# Step 1: Fetch candidate pool using required filters
print("\n── Step 1: Fetching candidate pool (required filters) ──")
base_ids, base_plants = fetch_plant_ids(session, REQUIRED_FILTERS, label="required")

# Deduplicate by post_id
seen = set()
unique_plants = []
for p in base_plants:
    if p["post_id"] not in seen:
        seen.add(p["post_id"])
        unique_plants.append(p)
base_plants = unique_plants
print(f"Candidate pool: {len(base_plants)} unique species")

# Step 2: Score candidates against preferred criteria
print("\n── Step 2: Scoring against preferred criteria ──")
composite, criterion_scores, roc_weights = compute_scores(
    session, REQUIRED_FILTERS, base_ids, PREFERRED_FILTERS
)

# Step 3: Generate output
print("\n── Step 3: Generating spreadsheet ──")
build_xlsx(
    base_plants, composite, criterion_scores, roc_weights,
    PREFERRED_FILTERS, REQUIRED_FILTERS, OUTPUT_FILE
)

# Summary statistics
scores = sorted(composite.values(), reverse=True)
n = len(scores)
print(f"\nResults: {n} plants scored")
print(f"  Score range: {min(scores):.1f} – {max(scores):.1f}")
print(f"  Mean: {sum(scores)/n:.1f}")
print(f"  ≥70 (high): {sum(1 for s in scores if s >= 70)}")
print(f"  40–69 (moderate): {sum(1 for s in scores if 40 <= s < 70)}")
print(f"  <40 (low): {sum(1 for s in scores if s < 40)}")

print(f"\n Downloading {OUTPUT_FILE}...")
files.download(OUTPUT_FILE)
print("Done.")

Session initialized. Nonce: be4fcc1fc4

── Step 1: Fetching candidate pool (required filters) ──
  92 plants across 10 page(s) [required]
Candidate pool: 92 unique species

── Step 2: Scoring against preferred criteria ──

Scoring model: 6 criteria, equal weights (1/6 = 0.1667 each):
  short_lived: values={'0': 1}
  good_for_novices: values={'easy': 2, 'average': 1}
  availability: values={'common': 2, 'variable': 1}
  water_requirements: values={('1.00', '2.00'): 1}
  spreader: values={'spreader': 2, 'variable': 1}
  pruning_needs: values={'regular': 2, 'minimal': 1}

Querying preferred criteria...
  11 plants across 2 page(s) [short_lived=0]
  3 plants across 1 page(s) [good_for_novices=easy]
  79 plants across 8 page(s) [good_for_novices=average]
  22 plants across 3 page(s) [availability=common]
  70 plants across 7 page(s) [availability=variable]
  70 plants across 7 page(s) [water_requirements=('1.00', '2.00')]
  14 plants across 2 page(s) [spreader=spreader]
  17 plants across 2

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done.
